In [1]:
import math

import ee
import geemap

import pandas as pd
import csv
import time
import os

/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [2]:
project_id = "ac215-bhargav"
ee.Authenticate()
ee.Initialize(project=project_id)

In [3]:
INPUT_CSV  = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/AGB_EO_EMIT.csv"
OUTPUT_CSV = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/AGB_EMIT_EO_HEIGHT.csv"

emit_df = pd.read_csv(INPUT_CSV)
#print(emit_df.columns)
print(f"emit_df height null count   : {emit_df['height'].isnull().sum()}")

emit_df height null count   : 0


# CODE

## COMMON FUNCTIONS

In [10]:
def build_points_from_group(group_df):
    point_features = []
    for _, row in group_df.iterrows():
        lon    = pd.to_numeric(row['longitude'], errors='coerce')
        lat    = pd.to_numeric(row['latitude'],  errors='coerce')
        row_id = int(row['row_id'])

        if pd.isna(lon) or pd.isna(lat):
            print(f"Skipping invalid coordinate: row_id={row_id}, lon={lon}, lat={lat}")
            continue

        point_features.append(
            ee.Feature(ee.Geometry.Point([lon, lat])).set({
                'row_id'  : row_id,
                'orig_lon': lon,
                'orig_lat': lat
            })
        )

    print(f"Valid points: {len(point_features)} / {len(group_df)}")
    return ee.FeatureCollection(point_features)

## SIMARD FUNCTIONS

In [11]:
def extract_simard_canopy_height(points):
    """
    Extract Simard et al. mangrove canopy height at each point location.
    Dataset : NASA/JPL/global_forest_canopy_height_2005
    Band    : '1' = canopy height in meters
    Resolution: 30m
    """
    simard = ee.Image("NASA/JPL/global_forest_canopy_height_2005")

    reduced = simard.select('1').reduceRegions(
        collection = points,
        reducer    = ee.Reducer.mean(),
        scale      = 30
    )

    def add_metadata(feature):
        return feature.set({
            'row_id'  : feature.get('row_id'),
            'orig_lon': feature.get('orig_lon'),
            'orig_lat': feature.get('orig_lat')
        })

    return reduced.map(add_metadata)

In [12]:
def create_simard_df(points):
    print("EXTRACTING SIMARD CANOPY HEIGHT")

    result = extract_simard_canopy_height(points)
    info   = result.getInfo()

    rows = []
    for feature in info['features']:
        props = feature['properties']
        rows.append({
            'row_id'          : props.get('row_id'),
            'longitude'       : props.get('orig_lon'),
            'latitude'        : props.get('orig_lat'),
            'simard_height_m' : props.get('mean')
        })

    df = pd.DataFrame(rows)

    print(f"Extracted rows    : {len(df)}")
    print(f"Null height values: {df['simard_height_m'].isnull().sum()}")

    if df['simard_height_m'].notna().any():
        print(f"Height min        : {df['simard_height_m'].min():.2f} m")
        print(f"Height max        : {df['simard_height_m'].max():.2f} m")
        print(f"Height mean       : {df['simard_height_m'].mean():.2f} m")

    return df

In [13]:
def run_simard(emit_df):
    # Load AGB data
    #agb_df              = pd.read_csv(input_csv)
    agb_df              = emit_df.reset_index(drop=True)
    agb_df['row_id']    = agb_df.index.astype(int)
    #agb_df['latitude']  = pd.to_numeric(agb_df['latitude'],  errors='coerce')
    #agb_df['longitude'] = pd.to_numeric(agb_df['longitude'], errors='coerce')
    #agb_df              = agb_df.dropna(subset=['latitude', 'longitude'])

    print(f"Total rows    : {len(agb_df)}")
    print(f"Unique plots  : {agb_df['plot_id'].nunique()}")

    # Build GEE points
    points     = build_points_from_group(agb_df)

    # Extract Simard height
    simard_df  = create_simard_df(points)

    return simard_df

## TANDEM-X FUNCTIONS

In [14]:
def extract_tandemx_canopy_height(points, buffer_meters=30):
    """
    Extract TanDEM-X mangrove canopy height at each point location.
    Dataset : projects/sat-io/open-datasets/GLOBAL_MANGROVE_HT_TANDEMX
    Resolution: 12m
    Non-mangrove pixels = 0
    """
    # Mosaic the image collection into a single image
    tandemx = ee.ImageCollection(
        "projects/sat-io/open-datasets/GLOBAL_MANGROVE_HT_TANDEMX"
    ).mosaic()

    # Buffer points to capture surrounding pixels
    buffered_points = points.map(
        lambda f: f.buffer(buffer_meters).copyProperties(f)
    )

    # Extract mean height within buffer
    reduced = tandemx.reduceRegions(
        collection = buffered_points,
        reducer    = ee.Reducer.mean(),
        scale      = 12
    )

    def add_metadata(feature):
        return feature.set({
            'row_id'  : feature.get('row_id'),
            'orig_lon': feature.get('orig_lon'),
            'orig_lat': feature.get('orig_lat')
        })

    return reduced.map(add_metadata)

In [15]:
def create_tandemx_df(points):
    print("EXTRACTING TANDEMX CANOPY HEIGHT")

    result = extract_tandemx_canopy_height(points)
    info   = result.getInfo()

    rows = []
    for feature in info['features']:
        props = feature['properties']
        height = props.get('mean')

        # Replace zero values with NaN — zero means non-mangrove pixel
        if height == 0:
            height = None

        rows.append({
            'longitude'        : props.get('orig_lon'),
            'latitude'         : props.get('orig_lat'),
            'tandemx_height_m' : height
        })

    df = pd.DataFrame(rows)

    print(f"Extracted rows         : {len(df)}")
    print(f"Null height values     : {df['tandemx_height_m'].isnull().sum()}")

    if df['tandemx_height_m'].notna().any():
        print(f"Height min             : {df['tandemx_height_m'].min():.2f} m")
        print(f"Height max             : {df['tandemx_height_m'].max():.2f} m")
        print(f"Height mean            : {df['tandemx_height_m'].mean():.2f} m")
        print(f"Unique height values   : {df['tandemx_height_m'].nunique()}")

    return df

In [16]:
def run_tandemx(emit_df):
    # Load AGB data
    #agb_df              = pd.read_csv(input_csv)
    agb_df              = emit_df.reset_index(drop=True)
    agb_df['row_id']    = agb_df.index.astype(int)
    #agb_df['latitude']  = pd.to_numeric(agb_df['latitude'],  errors='coerce')
    #agb_df['longitude'] = pd.to_numeric(agb_df['longitude'], errors='coerce')
    #agb_df              = agb_df.dropna(subset=['latitude', 'longitude'])

    print(f"Total rows    : {len(agb_df)}")
    print(f"Unique plots  : {agb_df['plot_id'].nunique()}")

    # Build GEE points
    points     = build_points_from_group(agb_df)

    # Extract Simard height
    tandemx_df  = create_tandemx_df(points)

    return tandemx_df

# EXTRACT SIMARD

In [17]:
simard_df = run_simard(emit_df)
simard_df['simard_height_m'].unique()

Total rows    : 3880
Unique plots  : 59
Valid points: 3880 / 3880
EXTRACTING SIMARD CANOPY HEIGHT
Extracted rows    : 3880
Null height values: 0
Height min        : 0.00 m
Height max        : 14.00 m
Height mean       : 3.97 m


array([10,  0,  9, 11, 13, 14,  7])

In [18]:
simard_dupes = simard_df.groupby(['latitude', 'longitude']).size()
print(f"Max rows per coordinate in simard_df : {simard_dupes.max()}")
print(f"Coordinates with duplicates          : {(simard_dupes > 1).sum()}")
print(simard_dupes[simard_dupes > 1].head(10))

simard_deduped = simard_df.groupby(
    ['latitude', 'longitude'], as_index=False
)['simard_height_m'].mean()

Max rows per coordinate in simard_df : 120
Coordinates with duplicates          : 59
latitude   longitude 
16.182060  -88.656500    105
16.182147  -88.656372    105
16.182230  -88.656190    117
16.182337  -88.656050     70
16.182404  -88.655890     90
16.182480  -88.655700    118
16.288337  -88.591185     81
16.288585  -88.591221    116
16.288840  -88.591300     76
16.289035  -88.591352     95
dtype: int64


In [19]:
print(f"Simard rows before dedup : {len(simard_df)}")
print(f"Simard rows after dedup  : {len(simard_deduped)}")
print(f"Simard height null count   : {simard_deduped['simard_height_m'].isnull().sum()}")

Simard rows before dedup : 3880
Simard rows after dedup  : 59
Simard height null count   : 0


# EXTRACT TANDEMX

In [20]:
tandemx_df = run_tandemx(emit_df)

Total rows    : 3880
Unique plots  : 59
Valid points: 3880 / 3880
EXTRACTING TANDEMX CANOPY HEIGHT
Extracted rows         : 3880
Null height values     : 68
Height min             : 1.02 m
Height max             : 20.40 m
Height mean            : 2.84 m
Unique height values   : 57


In [21]:
# Impute missing values
null_count = tandemx_df['tandemx_height_m'].isnull().sum()
print(f"TandemX height null count   : {null_count}")

if null_count:
    median_height = tandemx_df['tandemx_height_m'].median()
    tandemx_df['tandemx_height_m'] = tandemx_df['tandemx_height_m'].fillna(median_height)
print(f"tandemx_df height null count   : {tandemx_df['tandemx_height_m'].isnull().sum()}")

TandemX height null count   : 68
tandemx_df height null count   : 0


In [22]:
len(tandemx_df['tandemx_height_m'].unique())

57

In [23]:
tandemx_deduped = tandemx_df.groupby(
    ['latitude', 'longitude'], as_index=False
)['tandemx_height_m'].mean()

print(f"tandemx rows before dedup : {len(tandemx_df)}")
print(f"tandemx rows after dedup  : {len(tandemx_deduped)}")
print(f"tandemx_deduped height null count   : {tandemx_deduped['tandemx_height_m'].isnull().sum()}")

tandemx rows before dedup : 3880
tandemx rows after dedup  : 59
tandemx_deduped height null count   : 0


# Merge SIMARD and TANDEMX with EMIT

## Merge SIMARD with EMIT

In [24]:
merged_df_simard = emit_df.merge(
    simard_deduped[['longitude', 'latitude', 'simard_height_m']],
    on  = ['longitude', 'latitude'],
    how = 'left'
)

In [25]:
print(f"Rows before merge          : {len(emit_df)}")
print(f"Rows after merge           : {len(merged_df_simard)}")
print(f"Simard height null count   : {merged_df_simard['simard_height_m'].isnull().sum()}")

Rows before merge          : 3880
Rows after merge           : 3880
Simard height null count   : 0


## Merge TANDEMX with EMIT

In [26]:
final_merged_df = merged_df_simard.merge(
    tandemx_deduped[['longitude', 'latitude', 'tandemx_height_m']],
    on  = ['longitude', 'latitude'],
    how = 'left'
)

In [27]:
print(f"Rows before merge          : {len(merged_df_simard)}")
print(f"Rows after merge           : {len(final_merged_df)}")

Rows before merge          : 3880
Rows after merge           : 3880


In [28]:
print(f"Simard height null count    : {final_merged_df['simard_height_m'].isnull().sum()}")
print(f"TandemX height null count   : {final_merged_df['tandemx_height_m'].isnull().sum()}")

Simard height null count    : 0
TandemX height null count   : 0


In [29]:
assert len(final_merged_df["simard_height_m"].head())
assert len(final_merged_df["tandemx_height_m"].head())

## SAVE to CSV

In [30]:
final_merged_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved EMIT output to: {OUTPUT_CSV}")

Saved EMIT output to: ../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/AGB_EMIT_EO_HEIGHT.csv
